# Sprint 9 Slice 5 - Multi-Seed Runner (runner-only)

Runner-only notebook. It contains **no model logic**: it stages data, calls the
existing repo runners over the predeclared seeds in
`configs/sweeps/sprint9_multiseed.yaml`, writes only under
`outputs/sprint9/multiseed/`, validates the per-seed output contract, and copies
results to Drive (excluding `model.pt`).

**Prerequisites**
- Runtime -> Change runtime type -> **GPU** (T4/L4/A100).
- Google Drive project root with `data/raw/` and `data/processed/` (same layout the
  Sprint 8A/8B/7F notebooks use): `/content/drive/MyDrive/crispr_gnn_offtarget` or
  `/content/drive/MyDrive/crispr-gnn-offtarget`.
- F4 (XGBoost) is **CPU** and is run locally, not here (`RUN_F4_CPU=False`).

This trains 4 GNN configs x 5 seeds = **20 trainings**. Re-running skips seeds whose
output already exists, so the notebook is resumable.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Configuration. Edit GIT_REF if you push to a different branch.
import os
REPO_URL = 'https://github.com/YasinEkici/crispr-gnn-offtarget.git'
REPO_DIR = '/content/crispr-gnn-offtarget'
GIT_REF  = 'sprint9/robustness-uncertainty'
RUN_GNN     = True    # train the 4 GNN configs on GPU
RUN_F4_CPU  = False   # F4 is run locally on CPU, not on Colab
os.environ['REPO_URL'] = REPO_URL
os.environ['REPO_DIR'] = REPO_DIR
os.environ['GIT_REF']  = GIT_REF
print(REPO_URL, GIT_REF, 'RUN_GNN=', RUN_GNN, 'RUN_F4_CPU=', RUN_F4_CPU)


In [ ]:
%%bash
set -euo pipefail
if [ -d "$REPO_DIR/.git" ]; then
  cd "$REPO_DIR"
  git fetch origin "$GIT_REF"
  git checkout "$GIT_REF"
  git pull --ff-only origin "$GIT_REF"
else
  git clone --branch "$GIT_REF" "$REPO_URL" "$REPO_DIR"
  cd "$REPO_DIR"
fi
git rev-parse --short HEAD


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"
python -m pip install -q uv
uv sync
uv run python - <<'PY'
import torch, torch_geometric
print('torch', torch.__version__)
print('torch_geometric', torch_geometric.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
PY


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"
DRIVE_ROOT_CANDIDATES=(
  "${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
  "/content/drive/MyDrive/crispr-gnn-offtarget"
)
DRIVE_ROOT=""
for CANDIDATE in "${DRIVE_ROOT_CANDIDATES[@]}"; do
  if [ -d "$CANDIDATE" ]; then DRIVE_ROOT="$CANDIDATE"; break; fi
done
if [ -z "$DRIVE_ROOT" ]; then
  echo "No Drive project root found. Checked:" >&2
  printf "  %s\n" "${DRIVE_ROOT_CANDIDATES[@]}" >&2
  find /content/drive/MyDrive -maxdepth 1 -type d | sort >&2
  exit 1
fi
echo "Using DRIVE_ROOT=$DRIVE_ROOT"
mkdir -p data/raw data/processed
if [ -d "$DRIVE_ROOT/data/raw" ]; then rsync -a "$DRIVE_ROOT/data/raw/" data/raw/; else echo "Missing $DRIVE_ROOT/data/raw" >&2; exit 1; fi
if [ -d "$DRIVE_ROOT/data/processed" ]; then rsync -a "$DRIVE_ROOT/data/processed/" data/processed/; fi


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"
# Graph C S5F2 artifact required by all four GNN configs (7F/8A/8B).
if [ ! -d data/processed/graphs/sprint5b/graph_c_context_observation ] || [ ! -f data/processed/graphs/sprint5b/graph_c_context_observation/features_S5F2_energy.parquet ]; then
  echo "Building Sprint 5B Graph C S5F2 artifact..."
  uv run python scripts/build_sprint5b_graph_c_energy_features.py \
    --data-config configs/data/mak2022.yaml \
    --schema-config configs/sweeps/graph_schema_ablation.yaml \
    --source-artifact-dir data/processed/graphs/sprint3 \
    --artifact-dir data/processed/graphs/sprint5b \
    --report-path outputs/sprint5b/graph_c_energy_sensitivity_artifact_report.md
else
  echo "Graph C S5F2 artifact present."
fi


In [ ]:
# Orchestration: loop manifest runs x predeclared seeds, calling existing runners.
# Resumable: a seed whose comparison file already exists is skipped.
import subprocess
from pathlib import Path
import yaml

repo = Path(REPO_DIR)
manifest = yaml.safe_load((repo / 'configs/sweeps/sprint9_multiseed.yaml').read_text())
base = repo / manifest['output_root']

for run in manifest['runs']:
    if run['device'] == 'colab_gpu' and not RUN_GNN:
        continue
    if run['runner_type'] == 'xgb_f4' and not RUN_F4_CPU:
        continue
    for seed in manifest['seeds']:
        out_dir = base / run['output_prefix'] / f'seed_{seed}'
        done_file = out_dir / run['comparison_file']
        if done_file.exists() and done_file.stat().st_size > 0:
            print(f'SKIP (exists): {run["registry_id"]} seed {seed}')
            continue
        run_id = f"sprint9_multiseed_{run['registry_id']}_seed{seed}"
        if run['runner_type'] == 'xgb_f4':
            cmd = ['uv', 'run', 'python', run['runner'],
                   '--seed', str(seed), '--output-dir', str(out_dir)]
        else:
            cmd = ['uv', 'run', 'python', run['runner'],
                   '--config', run['config'], '--run', run['run_arg'],
                   '--run-id', run_id, '--seed', str(seed), '--output-dir', str(out_dir)]
        print('\nRUN:', ' '.join(cmd))
        subprocess.run(cmd, cwd=repo, check=True)


In [ ]:
# Validate the per-seed output contract before copying back.
missing = []
for run in manifest['runs']:
    if run['device'] == 'colab_gpu' and not RUN_GNN:
        continue
    if run['runner_type'] == 'xgb_f4' and not RUN_F4_CPU:
        continue
    for seed in manifest['seeds']:
        expected = base / run['output_prefix'] / f'seed_{seed}' / run['comparison_file']
        if not expected.exists() or expected.stat().st_size == 0:
            missing.append(str(expected.relative_to(repo)))
if missing:
    print('MISSING per-seed outputs:')
    for m in missing:
        print(' ', m)
else:
    print('OK: all expected per-seed outputs present for the selected scope.')


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_DIR"
DRIVE_ROOT_CANDIDATES=(
  "${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
  "/content/drive/MyDrive/crispr-gnn-offtarget"
)
DRIVE_ROOT=""
for CANDIDATE in "${DRIVE_ROOT_CANDIDATES[@]}"; do
  if [ -d "$CANDIDATE" ]; then DRIVE_ROOT="$CANDIDATE"; break; fi
done
test -n "$DRIVE_ROOT"
RETURN_ROOT="$DRIVE_ROOT/returned_outputs/sprint9_multiseed"
mkdir -p "$RETURN_ROOT"
# Copy only per-seed outputs; never copy model.pt (009 plan: no committed weights).
rsync -a --exclude="model.pt" --exclude=".DS_Store" outputs/sprint9/multiseed/ "$RETURN_ROOT"/
echo "Copied multiseed outputs to $RETURN_ROOT"
find "$RETURN_ROOT" -maxdepth 3 -name "*comparison*.csv" | sort


## After this notebook

1. Download/sync `returned_outputs/sprint9_multiseed/` from Drive back into the repo
   at `outputs/sprint9/multiseed/` (preserving `<output_prefix>/seed_<N>/` layout).
2. Locally run: `uv run python scripts/run_sprint9_robustness.py --stage multiseed`
3. That regenerates `outputs/sprint9/robustness_model_seed_summary.csv` and
   `outputs/sprint9/figures/per_seed_metric_variance.png` with the GNN seeds filled
   in alongside the already-present F4 seeds (no best-seed selection).
